# 🍎 Tutorial Training Model FreshScan di Google Colab
### Transfer Learning Xception — Klasifikasi Kesegaran Buah (Fresh vs Rotten)

Notebook ini melatih model yang akan dipakai oleh aplikasi Flask **FreshScan**.

**Alur singkat:**
1. Aktifkan GPU di Colab
2. Upload dataset yang sudah kamu unduh dari Kaggle
3. Konversi struktur dataset ke format biner `fresh` / `rotten`
4. Training model Xception (feature extraction + fine-tuning)
5. Download hasil (`xception_fruit_freshness.h5` + `class_indices.json`) untuk dipasang di folder `model/` pada project Flask

> ⚠️ **Sebelum mulai:** aktifkan GPU dulu supaya training cepat.
> Menu **Runtime → Change runtime type → Hardware accelerator → GPU (T4)** → Save.


## 1. Cek GPU aktif

Jalankan cell di bawah. Kalau hasilnya kosong / error, berarti GPU belum aktif — ulangi langkah di atas.


In [ ]:
# Mengecek apakah Colab sudah terhubung ke GPU
!nvidia-smi


## 2. Upload Dataset

Karena dataset sudah kamu unduh manual dari Kaggle, ada **2 cara** untuk memasukkannya ke Colab.
Pilih salah satu sesuai kondisimu.

### Opsi A — Upload langsung file ZIP (paling gampang jika ukuran dataset tidak terlalu besar)
Kamu perlu meng-compress folder dataset hasil download Kaggle menjadi satu file `.zip`
terlebih dahulu di komputer (klik kanan folder → Compress/Send to ZIP), lalu jalankan cell di bawah.

### Opsi B — Simpan di Google Drive lalu mount (disarankan jika dataset besar / mau dipakai berkali-kali)
Upload folder/zip dataset ke Google Drive kamu terlebih dahulu, lalu jalankan cell mount Drive.

Jalankan **salah satu** dari dua cell di bawah (A atau B), sesuai pilihanmu.


In [ ]:
# ============================
# OPSI A: Upload file ZIP dataset langsung dari komputer
# ============================
# Jalankan cell ini, lalu klik tombol "Choose Files" yang muncul,
# dan pilih file zip dataset Kaggle yang sudah kamu unduh
# (misal: fruits-fresh-and-rotten-for-classification.zip)

from google.colab import files
uploaded = files.upload()

# Menampilkan nama file yang berhasil diupload
for filename in uploaded.keys():
    print(f"File terupload: {filename}")


In [ ]:
# Ekstrak file zip yang baru diupload (Opsi A) ke folder 'dataset_raw'
import zipfile, os

zip_filename = list(uploaded.keys())[0]   # ambil nama file zip yang tadi diupload

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall("dataset_raw")

print("Ekstrak selesai. Isi folder dataset_raw:")
!ls dataset_raw


In [ ]:
# ============================
# OPSI B: Mount Google Drive (pakai ini kalau dataset disimpan di Drive)
# ============================
from google.colab import drive
drive.mount('/content/drive')

# Setelah ter-mount, sesuaikan path berikut dengan lokasi dataset di Drive kamu.
# Contoh jika kamu menyimpan zip di My Drive/datasets/fruits.zip:
DRIVE_ZIP_PATH = "/content/drive/MyDrive/datasets/fruits-fresh-and-rotten-for-classification.zip"

import zipfile
with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall("dataset_raw")

print("Ekstrak dari Drive selesai. Isi folder dataset_raw:")
!ls dataset_raw


## 3. Cek Struktur Dataset

Dataset asli dari Kaggle biasanya punya struktur seperti ini:
```
dataset_raw/
└── dataset/
    ├── train/
    │   ├── freshapples/
    │   ├── freshbanana/
    │   ├── freshoranges/
    │   ├── rottenapples/
    │   ├── rottenbanana/
    │   └── rottenoranges/
    └── test/
        └── (folder yang sama)
```
Jalankan cell di bawah untuk melihat struktur folder secara otomatis, lalu **sesuaikan**
variabel `SOURCE_DIR` pada cell berikutnya jika path-nya berbeda.


In [ ]:
# Menampilkan struktur folder dataset_raw sampai 3 level agar mudah dicek
import os

for root, dirs, files_ in os.walk("dataset_raw"):
    level = root.replace("dataset_raw", "").count(os.sep)
    if level > 2:
        continue
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")


## 4. Konversi ke Struktur Biner (Fresh / Rotten)

Model kita hanya mengklasifikasikan **2 kelas**: `fresh` dan `rotten` (menggabungkan
apel, pisang, dan jeruk menjadi satu label besar). Cell di bawah otomatis membaca
semua folder yang mengandung kata `fresh` atau `rotten` di namanya, lalu menyalin
gambarnya ke struktur baru:

```
data/
├── train/
│   ├── fresh/
│   └── rotten/
└── test/
    ├── fresh/
    └── rotten/
```

**Penting:** sesuaikan `SOURCE_DIR` di bawah ini dengan path hasil ekstrak pada langkah
sebelumnya (lihat output struktur folder di atas untuk menentukan path yang tepat,
biasanya `dataset_raw/dataset` atau langsung `dataset_raw`).


In [ ]:
# ------------------------------- KONFIGURASI PATH -------------------------------
# GANTI path ini sesuai hasil pengecekan struktur folder di langkah 3
SOURCE_DIR = "dataset_raw/dataset"   # <-- sesuaikan jika perlu
TARGET_DIR = "data"

print(f"Sumber dataset : {SOURCE_DIR}")
print(f"Target dataset : {TARGET_DIR}")


In [ ]:
# ------------------------------- SCRIPT KONVERSI DATASET -------------------------------
# Fungsi ini setara dengan dataset_tools/prepare_dataset.py pada project Flask,
# dijalankan langsung di Colab untuk kemudahan.

import shutil
from pathlib import Path

FRESH_KEYWORD = "fresh"
ROTTEN_KEYWORD = "rotten"

def classify_folder_name(folder_name):
    """Menentukan label fresh/rotten berdasarkan nama folder kelas asli."""
    name_lower = folder_name.lower()
    if ROTTEN_KEYWORD in name_lower:
        return "rotten"
    if FRESH_KEYWORD in name_lower:
        return "fresh"
    return None

def copy_split(source_split_dir: Path, target_split_dir: Path):
    """Menyalin gambar dari satu split (train/test) ke struktur biner."""
    (target_split_dir / "fresh").mkdir(parents=True, exist_ok=True)
    (target_split_dir / "rotten").mkdir(parents=True, exist_ok=True)

    total_copied = 0
    for class_folder in sorted(source_split_dir.iterdir()):
        if not class_folder.is_dir():
            continue
        label = classify_folder_name(class_folder.name)
        if label is None:
            print(f"  [SKIP] {class_folder.name}")
            continue

        destination = target_split_dir / label
        count = 0
        for img_file in class_folder.glob("*"):
            if img_file.is_file():
                new_name = f"{class_folder.name}_{img_file.name}"
                shutil.copy2(img_file, destination / new_name)
                count += 1
        total_copied += count
        print(f"  [OK] {class_folder.name:20s} -> {label} ({count} gambar)")

    print(f"Total gambar disalin ke '{target_split_dir}': {total_copied}\n")


source_root = Path(SOURCE_DIR)
target_root = Path(TARGET_DIR)

for split_name in ["train", "test"]:
    source_split = source_root / split_name
    if not source_split.exists():
        print(f"[PERINGATAN] Split '{split_name}' tidak ditemukan, dilewati.")
        continue
    print(f"Memproses split: {split_name}")
    copy_split(source_split, target_root / split_name)

print("Konversi selesai! Struktur dataset biner ada di folder:", target_root.resolve())


In [ ]:
# Cek jumlah gambar akhir per kelas, memastikan konversi berhasil
import os

for split in ["train", "test"]:
    for label in ["fresh", "rotten"]:
        path = f"data/{split}/{label}"
        if os.path.exists(path):
            print(f"{path}: {len(os.listdir(path))} gambar")
        else:
            print(f"{path}: TIDAK DITEMUKAN")


## 5. Install & Import Library

Colab biasanya sudah menyediakan TensorFlow, tapi kita pastikan versinya sesuai.


In [ ]:
# Memastikan TensorFlow tersedia (Colab umumnya sudah preinstall)
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices('GPU'))


In [ ]:
# ------------------------------- IMPORT LIBRARY -------------------------------------
import os
import json
import matplotlib.pyplot as plt

from tensorflow.keras.applications import Xception
from tensorflow.keras.applications.xception import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam


## 6. Konfigurasi & Data Generator

Sama seperti `train_model.py` pada project Flask: 299×299 (ukuran wajib Xception),
augmentasi untuk data training, 20% dari data train dipakai sebagai validasi.


In [ ]:
# ------------------------------- KONFIGURASI DASAR -----------------------------------
IMG_SIZE = (299, 299)
BATCH_SIZE = 32
EPOCHS_FEATURE_EXTRACTION = 12
EPOCHS_FINE_TUNING = 8
VALIDATION_SPLIT = 0.2

TRAIN_DIR = "data/train"
TEST_DIR = "data/test"

MODEL_DIR = "model"
REPORT_DIR = "reports"
MODEL_PATH = os.path.join(MODEL_DIR, "xception_fruit_freshness.h5")
CLASS_INDEX_PATH = os.path.join(MODEL_DIR, "class_indices.json")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)


In [ ]:
# ------------------------------- DATA GENERATOR -----------------------------------
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=(0.8, 1.2),
    validation_split=VALIDATION_SPLIT,
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", subset="training", shuffle=True, seed=42,
)

validation_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", subset="validation", shuffle=False, seed=42,
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", shuffle=False,
)

print("Label mapping:", train_generator.class_indices)

# Simpan mapping label agar sesuai dengan yang dibaca app.py nanti
with open(CLASS_INDEX_PATH, "w") as f:
    json.dump(train_generator.class_indices, f)


## 7. Bangun Model (Transfer Learning Xception)

Base model Xception (pretrained ImageNet) dibekukan dahulu, lalu ditambahkan
classifier head baru untuk klasifikasi biner Fresh/Rotten.


In [ ]:
# ------------------------------- BANGUN MODEL -----------------------------------
base_model = Xception(weights="imagenet", include_top=False, input_shape=(299, 299, 3))
base_model.trainable = False   # bekukan seluruh layer Xception dahulu

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
x = Dense(64, activation="relu")(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()


## 8. Tahap 1 — Training Feature Extraction

Hanya classifier head yang dilatih, base Xception tetap dibekukan.


In [ ]:
# ------------------------------- CALLBACKS -----------------------------------
callbacks = [
    ModelCheckpoint(MODEL_PATH, monitor="val_accuracy", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7),
]

history_stage1 = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS_FEATURE_EXTRACTION,
    callbacks=callbacks,
)


In [ ]:
# Menyimpan grafik akurasi & loss tahap 1 (untuk lampiran laporan)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_stage1.history["accuracy"], label="Train Accuracy")
axes[0].plot(history_stage1.history["val_accuracy"], label="Validation Accuracy")
axes[0].set_title("Akurasi Model - Tahap 1")
axes[0].legend()

axes[1].plot(history_stage1.history["loss"], label="Train Loss")
axes[1].plot(history_stage1.history["val_loss"], label="Validation Loss")
axes[1].set_title("Loss Model - Tahap 1")
axes[1].legend()

fig.tight_layout()
fig.savefig(f"{REPORT_DIR}/training_stage1_feature_extraction.png")
plt.show()


## 9. Tahap 2 — Fine-Tuning

Membuka (unfreeze) 30 layer terakhir Xception agar model lebih spesifik terhadap
karakteristik dataset buah, dengan learning rate yang lebih kecil.


In [ ]:
# ------------------------------- FINE-TUNING -----------------------------------
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

history_stage2 = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS_FINE_TUNING,
    callbacks=callbacks,
)


In [ ]:
# Menyimpan grafik akurasi & loss tahap 2 (fine-tuning)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_stage2.history["accuracy"], label="Train Accuracy")
axes[0].plot(history_stage2.history["val_accuracy"], label="Validation Accuracy")
axes[0].set_title("Akurasi Model - Tahap 2 (Fine-Tuning)")
axes[0].legend()

axes[1].plot(history_stage2.history["loss"], label="Train Loss")
axes[1].plot(history_stage2.history["val_loss"], label="Validation Loss")
axes[1].set_title("Loss Model - Tahap 2 (Fine-Tuning)")
axes[1].legend()

fig.tight_layout()
fig.savefig(f"{REPORT_DIR}/training_stage2_fine_tuning.png")
plt.show()


## 10. Evaluasi Akhir & Simpan Model

Uji model pada `test_generator` (data yang sama sekali belum pernah dilihat model).


In [ ]:
# ------------------------------- EVALUASI PADA DATA TEST -----------------------------------
test_loss, test_acc = model.evaluate(test_generator)
print(f"Akurasi pada data test : {test_acc:.4f}")
print(f"Loss pada data test    : {test_loss:.4f}")

# Simpan model final
model.save(MODEL_PATH)
print(f"Model tersimpan di: {MODEL_PATH}")


In [ ]:
# (Opsional tapi direkomendasikan) Confusion matrix & classification report
# untuk dilampirkan di laporan penelitian
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

test_generator.reset()
y_pred_prob = model.predict(test_generator)
y_pred = (y_pred_prob >= 0.5).astype(int).flatten()
y_true = test_generator.classes

label_names = list(test_generator.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=label_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=label_names, yticklabels=label_names)
plt.xlabel("Prediksi")
plt.ylabel("Label Asli")
plt.title("Confusion Matrix")
plt.savefig(f"{REPORT_DIR}/confusion_matrix.png")
plt.show()


## 11. Download Hasil Training ke Komputer

Download 2 file berikut, lalu **letakkan di folder `model/`** pada project Flask
`fruit-freshness-app` di komputermu:
- `xception_fruit_freshness.h5`
- `class_indices.json`

Setelah itu, jalankan `python app.py` seperti biasa — aplikasi akan otomatis
mendeteksi dan memuat model ini.


In [ ]:
# Download file model & mapping label ke komputer lokal
from google.colab import files

files.download(MODEL_PATH)
files.download(CLASS_INDEX_PATH)

# (Opsional) download juga grafik-grafik hasil training untuk laporan
files.download(f"{REPORT_DIR}/training_stage1_feature_extraction.png")
files.download(f"{REPORT_DIR}/training_stage2_fine_tuning.png")
files.download(f"{REPORT_DIR}/confusion_matrix.png")


## 12. (Opsional) Simpan Model ke Google Drive

Kalau kamu tidak mau bolak-balik download manual, simpan langsung ke Drive:


In [ ]:
# ------------------------------- SIMPAN KE GOOGLE DRIVE (OPSIONAL) -----------------------------------
# Pastikan drive sudah di-mount (lihat Opsi B di langkah 2). Jika belum:
# from google.colab import drive
# drive.mount('/content/drive')

import shutil

DRIVE_SAVE_DIR = "/content/drive/MyDrive/FreshScan_Model"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

shutil.copy(MODEL_PATH, DRIVE_SAVE_DIR)
shutil.copy(CLASS_INDEX_PATH, DRIVE_SAVE_DIR)

print(f"Model tersimpan di Google Drive: {DRIVE_SAVE_DIR}")
